In [2]:
import json
import os
import uuid
from collections import Counter
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

from pinecone import Pinecone, ServerlessSpec
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
import os
from pathlib import Path
from dotenv import load_dotenv



load_dotenv()  # loads PINECONE_API_KEY from .env

try:
    from langchain_huggingface import HuggingFaceEmbeddings
except Exception:
    # fallback for older LangChain installs
    from langchain.embeddings import HuggingFaceEmbeddings


In [6]:
import json
from pathlib import Path
from typing import Any, Dict, List

def load_poc_json(path: str = "data/test_data") -> List[Dict[str, Any]]:
    """
    Load all POC records from all .json files in test_data folder.
    """
    path_obj = Path(path)
    
    if not path_obj.is_dir():
        raise FileNotFoundError(f"Directory '{path}' not found.")
    
    all_records: List[Dict[str, Any]] = []
    json_files = sorted(path_obj.glob("*.json"))
    
    if not json_files:
        raise FileNotFoundError(f"No .json files found in '{path}'")
    
    for json_file in json_files:
        try:
            with open(json_file, "r", encoding="utf-8") as f:
                raw = json.load(f)
            
            # Parse records from JSON structure
            if isinstance(raw, dict) and "data" in raw and isinstance(raw["data"], list):
                records = raw["data"]
            elif isinstance(raw, dict):
                records = [raw]
            elif isinstance(raw, list):
                records = raw
            else:
                raise ValueError("Unsupported JSON structure.")
            
            all_records.extend(records)
            print(f"✓ {json_file.name}: {len(records)} records loaded")
        
        except Exception as e:
            print(f"✗ {json_file.name}: {e}")
    
    print(f"\n✓ Total: {len(all_records)} records from {len(json_files)} files\n")
    return all_records

json_data = load_poc_json(r"/home/labuser/Downloads/intel/frontend/uploads")

✓ 33174593.json: 1 records loaded
✓ f716dce1.json: 1 records loaded
✗ poc1.json: Expecting value: line 1 column 1 (char 0)

✓ Total: 2 records from 3 files



In [7]:

    # 1) Input JSON under data folder - CHANGE THIS PATH
str(json_data)



"[{'id': '33174593', 'title': 'Chatbot', 'description': 'creationi of chatbot on shopping app', 'problem': 'Creating user friendly interface using chatbot\\n', 'outcome': 'chatbifdsg', 'language': 'pyhon ', 'approach': 'RNN', 'stack': 'KNN', 'complexity': 'Low', 'boilerplate_enabled': False, 'dev_count': 2, 'skills': ['Frontend', 'Backend'], 'timeline': '80', 'manager': 'ram'}, {'id': 'f716dce1', 'title': 'cguabcijdasbciuw', 'description': 'kncodnowncwecwec', 'problem': 'wncoinnew9', 'outcome': 'ownevoiwenviwenvewov', 'language': 'pyhon cjbnweifnwif', 'approach': 'wbucibwei9cbuwebcuiwebwew', 'stack': 'nciowenciowecnweic', 'complexity': 'Medium', 'boilerplate_enabled': False, 'dev_count': 1, 'skills': ['Frontend', 'Data Engineer'], 'timeline': '80', 'manager': 'ram'}]"

In [8]:
def select_columns(records: List[Dict[str, Any]], include_columns: Sequence[str]) -> List[Dict[str, Any]]:
    """
    Keep only selected columns. Missing keys are skipped.
    """
    include_set = set(include_columns)
    return [{k: v for k, v in r.items() if k in include_set} for r in records]
include_columns = [  "id","title", "description", "problem", "outcome", "language", "approach", "stack", "skills",  ]

columns_json = select_columns(json_data,include_columns)



In [9]:


def chunk_documents(json_data: List[Dict[str, Any]], chunk_size: int = 500, chunk_overlap: int = 20) -> List[Document]:
    """
    json_data: list of POC dicts from load_poc_json()
    """
    if not isinstance(json_data, list):
        raise ValueError("chunk_documents expects json_data as list[dict].")

    docs: List[Document] = []
    for i, rec in enumerate(json_data):
        if not isinstance(rec, dict):
            continue

        lines = []
        for k, v in rec.items():
            if v is None:
                continue
            if isinstance(v, list):
                v = ", ".join(str(x) for x in v)
            lines.append(f"{k}: {v}")

        text = "\n".join(lines).strip()
        if not text:
            continue

        docs.append(
            Document(
                page_content=text,
                metadata={
                    "poc_id": str(rec.get("id", f"row_{i}")),
                    "chunk_source": "load_poc_json",
                    "raw_record": json.dumps(rec, ensure_ascii=False),  # Pinecone-safe (string)
                },
            )
        )

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_documents(docs)

    for idx, c in enumerate(chunks):
        c.metadata = dict(c.metadata or {})
        c.metadata["chunk_id"] = idx

    return chunks



In [10]:

chunks = chunk_documents(columns_json)

In [11]:
str(chunks)

'[Document(page_content=\'id: 33174593\\ntitle: Chatbot\\ndescription: creationi of chatbot on shopping app\\nproblem: Creating user friendly interface using chatbot\\n\\noutcome: chatbifdsg\\nlanguage: pyhon \\napproach: RNN\\nstack: KNN\\nskills: Frontend, Backend\', metadata={\'poc_id\': \'33174593\', \'chunk_source\': \'load_poc_json\', \'raw_record\': \'{"id": "33174593", "title": "Chatbot", "description": "creationi of chatbot on shopping app", "problem": "Creating user friendly interface using chatbot\\\\n", "outcome": "chatbifdsg", "language": "pyhon ", "approach": "RNN", "stack": "KNN", "skills": ["Frontend", "Backend"]}\', \'chunk_id\': 0}), Document(page_content=\'id: f716dce1\\ntitle: cguabcijdasbciuw\\ndescription: kncodnowncwecwec\\nproblem: wncoinnew9\\noutcome: ownevoiwenviwenvewov\\nlanguage: pyhon cjbnweifnwif\\napproach: wbucibwei9cbuwebcuiwebwew\\nstack: nciowenciowecnweic\\nskills: Frontend, Data Engineer\', metadata={\'poc_id\': \'f716dce1\', \'chunk_source\': \'l

In [26]:
chunks

[Document(page_content='id: 33174593\ntitle: Chatbot\ndescription: creationi of chatbot on shopping app\nproblem: Creating user friendly interface using chatbot\n\noutcome: chatbifdsg\nlanguage: pyhon \napproach: RNN\nstack: KNN\nskills: Frontend, Backend', metadata={'poc_id': '33174593', 'chunk_source': 'load_poc_json', 'raw_record': '{"id": "33174593", "title": "Chatbot", "description": "creationi of chatbot on shopping app", "problem": "Creating user friendly interface using chatbot\\n", "outcome": "chatbifdsg", "language": "pyhon ", "approach": "RNN", "stack": "KNN", "skills": ["Frontend", "Backend"]}', 'chunk_id': 0}),
 Document(page_content='id: f716dce1\ntitle: cguabcijdasbciuw\ndescription: kncodnowncwecwec\nproblem: wncoinnew9\noutcome: ownevoiwenviwenvewov\nlanguage: pyhon cjbnweifnwif\napproach: wbucibwei9cbuwebcuiwebwew\nstack: nciowenciowecnweic\nskills: Frontend, Data Engineer', metadata={'poc_id': 'f716dce1', 'chunk_source': 'load_poc_json', 'raw_record': '{"id": "f716dc

In [27]:
#embedding 
model_name = "sentence-transformers/all-MiniLM-L6-v2"
from langchain.embeddings import HuggingFaceEmbeddings
def download_embeddings():
    model_name ="sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name = model_name
        
    )
    return embeddings
embeddings = download_embeddings()

/home/labuser/Downloads/intel/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [28]:
import os
from  dotenv import load_dotenv
from  pinecone import Pinecone
load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
#OPEN_API_KEY = os.getenv("OPEN_API_KEY")


os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
#os.environ["OPEN_API_KEY"] =  OPEN_API_KEY


pinecone_api_key =PINECONE_API_KEY
pc = Pinecone(api_key= pinecone_api_key)

In [29]:
#Download the Embeddings from HuggingFace 
import os
from  dotenv import load_dotenv
load_dotenv()
from pinecone import Pinecone
from pinecone import ServerlessSpec 
from langchain_pinecone import PineconeVectorStore

def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')  #this model return 384 dimensions
    return embeddings
embeddings = download_hugging_face_embeddings()

In [32]:



index_name = "innoscan"  # change if desired
target_dim = 384


def index_exists(client: Pinecone, name: str) -> bool:
    indexes = client.list_indexes()
    if hasattr(indexes, "names"):  # newer SDK response object
        return name in indexes.names()
    # older/alternate response formats
    return name in [
        i["name"] if isinstance(i, dict) else getattr(i, "name", None)
        for i in indexes
    ]




In [35]:
#####  Create New index 
docsearch = PineconeVectorStore.from_documents(
    documents=chunks,
    index_name=index_name,
    embedding=embeddings, 
)

In [113]:

# Compare embedding dimension vs Pinecone index dimension
def get_index_dimension(pc, index_name: str):
    # Works across SDK response variants
    desc = pc.describe_index(index_name)
    if isinstance(desc, dict):
        return desc.get("dimension")
    return getattr(desc, "dimension", None)

def check_dimension_mismatch(pc, index_name: str, embeddings_model):
    # 1) embedding dimension
    sample_vec = embeddings_model.embed_query("dimension check")
    emb_dim = len(sample_vec)

    # 2) index dimension
    idx_dim = get_index_dimension(pc, index_name)

    print(f"Embedding dim: {emb_dim}")
    print(f"Index dim: {idx_dim}")

    if idx_dim is None:
        print("Could not read index dimension from API response.")
        return False

    if emb_dim != idx_dim:
        print("❌ Dimension mismatch detected.")
        return False

    print("✅ Dimensions match.")
    return True

ok = check_dimension_mismatch(pc, "innoscan", embeddings)
if not ok:
    # recreate index with correct dimension
    pass

Embedding dim: 384
Index dim: 384
✅ Dimensions match.


In [36]:
######  Load existing index
docsearch = PineconeVectorStore.from_documents(
    documents=chunks,
    index_name=index_name,
    embedding=embeddings, 
)

docsearch.add_documents(chunks)

['b22a47e0-d0cd-4b71-939a-c2aa098c38f5',
 '0026679e-4fe8-41df-9897-a8c0381775a3']

In [37]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})
retrieved_docs = retriever.invoke("What is POC ?")
retrieved_docs

[Document(page_content='id: poc_010\ntitle: Automated Compliance Checker\ndescription: Scan documents or code to check compliance with company policies.\nproblem: Manual compliance checks are slow and error-prone.\noutcome: Instant compliance scoring and recommendations.\nlanguage: Python\napproach: Embed policies + use vector search for compliance checking.\nstack: Python, LangChain, Elasticsearch, Gradio\nskills: Frontend, Data Engineer', metadata={'chunk_id': 1.0, 'chunk_source': 'load_poc_json', 'poc_id': 'poc_010', 'raw_record': '{"id": "poc_010", "title": "Automated Compliance Checker", "description": "Scan documents or code to check compliance with company policies.", "problem": "Manual compliance checks are slow and error-prone.", "outcome": "Instant compliance scoring and recommendations.", "language": "Python", "approach": "Embed policies + use vector search for compliance checking.", "stack": "Python, LangChain, Elasticsearch, Gradio", "skills": ["Frontend", "Data Engineer"]

In [ ]:
#### Rough code
# ==========================================
# Phase 4: Match-check + 70% threshold
# ==========================================

def query_similar(
    index,
    embeddings: HuggingFaceEmbeddings,
    query_text: str,
    top_k: int = 5,
    namespace: str = "default",
    include_metadata: bool = True
) -> List[Dict[str, Any]]:
    qvec = embeddings.embed_query(query_text)
    res = index.query(
        vector=qvec,
        top_k=top_k,
        namespace=namespace,
        include_values=False,
        include_metadata=include_metadata,
    )

    matches = getattr(res, "matches", None)
    if matches is None and isinstance(res, dict):
        matches = res.get("matches", [])
    matches = matches or []

    normalized: List[Dict[str, Any]] = []
    for m in matches:
        if isinstance(m, dict):
            normalized.append(
                {
                    "id": m.get("id"),
                    "score": float(m.get("score", 0.0)),
                    "metadata": m.get("metadata", {}),
                }
            )
        else:
            normalized.append(
                {
                    "id": getattr(m, "id", None),
                    "score": float(getattr(m, "score", 0.0)),
                    "metadata": getattr(m, "metadata", {}) or {},
                }
            )
    return normalized


def filter_by_threshold(matches: List[Dict[str, Any]], threshold: float = 0.70) -> List[Dict[str, Any]]:
    return [m for m in matches if m.get("score", 0.0) >= threshold]


def check_poc_match(
    index,
    embeddings: HuggingFaceEmbeddings,
    new_poc_record: Dict[str, Any],
    include_columns: Sequence[str],
    threshold: float = 0.70,
    top_k: int = 5,
    namespace: str = "default"
) -> Dict[str, Any]:
    normalized = normalize_record(new_poc_record)
    query_text = build_text_for_embedding(normalized, include_columns)
    if not query_text:
        return {
            "is_match": False,
            "reason": "No usable text from selected columns.",
            "query_text": "",
            "matches": [],
            "matched_details": [],
        }

    matches = query_similar(
        index=index,
        embeddings=embeddings,
        query_text=query_text,
        top_k=top_k,
        namespace=namespace,
        include_metadata=True,
    )
    passed = filter_by_threshold(matches, threshold=threshold)

    return {
        "is_match": len(passed) > 0,
        "threshold": threshold,
        "query_text": query_text,
        "matches": matches,
        "matched_details": show_matched_poc_details(passed),
    }


def show_matched_poc_details(matches: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    out: List[Dict[str, Any]] = []
    for m in matches:
        md = m.get("metadata", {}) or {}
        out.append(
            {
                "match_id": m.get("id"),
                "score": m.get("score"),
                "poc_id": md.get("poc_id"),
                "chunk_id": md.get("chunk_id"),
                "text": md.get("text"),
                "raw_record": md.get("raw_record", {}),
            }
        )
    return out


def debug_query_report(matches: List[Dict[str, Any]], threshold: float = 0.70) -> Dict[str, Any]:
    return {
        "retrieved": len(matches),
        "above_threshold": len([m for m in matches if m.get("score", 0.0) >= threshold]),
        "threshold": threshold,
        "top_scores": [round(m.get("score", 0.0), 4) for m in matches[:5]],
    }


# ==========================================
# Optional orchestration helpers
# ==========================================

def create_and_load_embeddings_from_json(
    json_path: str,
    include_columns: Sequence[str],
    index_name: str,
    namespace: str = "default",
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
    chunk_size: int = 500,
    chunk_overlap: int = 20,
    cloud: str = "aws",
    region: str = "us-east-1",
) -> Dict[str, Any]:
    records = load_poc_json(json_path)
    docs = records_to_documents(records, include_columns=include_columns)
    chunks = chunk_documents(docs, chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    embeddings = get_huggingface_embeddings(model_name=model_name)
    payloads = build_vector_payloads(chunks, embeddings=embeddings)

    if not payloads:
        return {"upserted_count": 0, "reason": "No payloads generated."}

    dim = len(payloads[0]["values"])
    pc = get_pinecone_client()
    index = ensure_index(pc, index_name=index_name, dimension=dim, cloud=cloud, region=region)
    upsert_info = upsert_payloads(index=index, payloads=payloads, namespace=namespace)

    return {
        "upsert_info": upsert_info,
        "schema_debug": debug_schema(records),
        "chunk_debug": debug_chunk_report(chunks),
        "vector_debug": debug_vector_report(payloads),
    }